In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
## for the first-time execution of this code
## install the packages in google Drive locally (/content/drive/MyDrive/Colab Notebooks/antigen_antibody/packages/)
import os
import sys

nb_path = '/content/notebooks'

if not os.path.exists(nb_path):
    os.symlink('/content/drive/MyDrive/@antibody/antigen_epitope/packages', nb_path)

sys.path.insert(0, nb_path)

In [ ]:
# first-time use execution
!pip install biopython --target=$nb_path

In [ ]:
!pip install scipy --target=$nb_path

In [ ]:
## non-first-time execution
## package was previously installed through process_data_positive_patch_3points-Graph.ipynb
import sys
sys.path.append('/content/drive/MyDrive/@antibody/antigen_epitope/packages')

In [ ]:
import os
import numpy as np
import pandas as pd
from Bio import PDB

In [ ]:
!apt-get install -y python3-pymol

In [ ]:
#@title TEST -- secondary structure
from pymol import cmd, stored

# Initialize an empty list to store secondary structure data in the stored namespace
stored.ss_values = []

# Load the PDB file into PyMOL
pdb_file_path = "/content/drive/MyDrive/@antibody/antigen_epitope/35.pdb"
cmd.load(pdb_file_path, "protein_complex")

# Select chain X
cmd.select("all_protein", "protein_complex and protein")

# Use cmd.iterate to access the 'ss' (secondary structure) attribute of each residue
cmd.iterate("all_protein and name CA", "stored.ss_values.append((resi, ss, chain))")


# Print the secondary structure for each residue
#for resi, ss in stored.ss_values:
#    print(f"Residue {resi}: Secondary Structure = {ss}")

stored.ss_values = []
cmd.iterate("all_protein and name CA", "stored.ss_values.append(ss)")
print(stored.ss_values)

In [ ]:
import matplotlib.pyplot as plt

# Sample secondary structure list
ss = ['L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'L', 'L', 'L', 'L', 'L', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'L', 'L']  # Your list

# Map colors
color_map = {
    'H': 'red',
    'L': 'cyan'
}

colors = [color_map[s] for s in ss]

# Plot
plt.figure(figsize=(20, 3))
plt.scatter(range(len(ss)), [1] * len(ss), c=colors, marker='s', s=50)
plt.yticks([])  # Hide y-ticks
plt.xlabel("Residue Index")
plt.title("DSSP Secondary Structure")

# Create a custom legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color='red', marker='s', label='Helix', markersize=10),
                   Line2D([0], [0], color='cyan', marker='s', label='Loop', markersize=10)]
plt.legend(handles=legend_elements, loc="upper left")

plt.tight_layout()
plt.show()


In [ ]:
#@title TEST -- secondary structure
from pymol import cmd, stored

# Initialize PyMOL (this might not be necessary or might not work in all environments)
#cmd.initialize()

# Initialize an empty list to store secondary structure data in the stored namespace
stored.ss_values = []

# Load the PDB file into PyMOL
pdb_file_path = "/content/drive/MyDrive/@antibody/antigen_epitope/35.pdb"
cmd.load(pdb_file_path, "protein_complex")

# Select all protein residues
cmd.select("all_protein", "protein_complex and protein")

# Let's debug by counting the number of residues in our selection
num_residues = cmd.count_atoms("all_protein and name CA")
print(f"Number of residues: {num_residues}")

# If we have residues, then proceed to fetch the secondary structure
if num_residues > 0:
    cmd.iterate("all_protein and name CA", "stored.ss_values.append((resi, ss, chain))")
    for resi, ss, chain in stored.ss_values:
        print(f"Chain {chain}, Residue {resi}: Secondary Structure = {ss}")
else:
    print("No protein residues found!")


In [ ]:
#@title TEST -- extract the antigen chain C and add SaSaRate to the b-factor of each residue

from Bio.PDB import PDBParser, PDBIO

# Initialize the PDB parser and read the file
parser = PDBParser(QUIET=True)
structure = parser.get_structure('protein_complex', '/content/drive/MyDrive/@antibody/antigen_epitope/data/structures/8goc_M_N_C.pdb')

# Create an empty Structure to save the chain C
from Bio.PDB.Structure import Structure
chain_C_structure = Structure('chain_C_structure')

# Add a model to the new structure
from Bio.PDB.Model import Model
model = Model(0)
chain_C_structure.add(model)

# Get chain C from the original structure and add it to the new structure
chain_C = structure[0]['C']
model.add(chain_C)

# Update B-factors based on the SaSaRate (you should already have this from your PyMOL calculations)
# Here, we're assuming SaSaRate is a dictionary with keys being the residue numbers and values being the calculated SASA
# SaSaRate = {...}

for residue in chain_C:
    resi = residue.get_id()[1]
    sasa_value = SaSaRate.get(('protein_complex', '', 'C', str(resi)), None)
    if sasa_value is not None:
        for atom in residue:
            atom.set_bfactor(sasa_value)

# Save the new structure as a PDB file
io = PDBIO()
io.set_structure(chain_C_structure)
io.save('/content/drive/MyDrive/@antibody/antigen_epitope/data/structures_antigen/8goc_C.pdb')
